## ***Assignment : Aggregations***

In [7]:
import sqlite3
import pandas as pd

# 1. Setup the 'World' Database environment in Memory
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Create tables similar to the MySQL 'World' database
cursor.executescript("""
CREATE TABLE country (
    Code TEXT PRIMARY KEY, Name TEXT, Continent TEXT, Region TEXT,
    Population INT, GNP REAL, LifeExpectancy REAL
);
CREATE TABLE city (
    ID INTEGER PRIMARY KEY, Name TEXT, CountryCode TEXT, Population INT
);
CREATE TABLE countrylanguage (
    CountryCode TEXT, Language TEXT, IsOfficial TEXT
);

-- Inserting sample data to make the queries work
INSERT INTO country VALUES
('USA', 'United States', 'North America', 'North America', 331000000, 21433000, 78.5),
('CAN', 'Canada', 'North America', 'North America', 38000000, 1647000, 82.3),
('IND', 'India', 'Asia', 'Southern Asia', 1380000000, 2875000, 69.7),
('CHN', 'China', 'Asia', 'Eastern Asia', 1440000000, 14342000, 76.9),
('FRA', 'France', 'Europe', 'Western Europe', 67000000, 2715000, 82.7),
('DEU', 'Germany', 'Europe', 'Western Europe', 83000000, 3861000, 81.3),
('BRA', 'Brazil', 'South America', 'South America', 212000000, 1847000, 75.9);

INSERT INTO city VALUES
(1, 'New York', 'USA', 8336000), (2, 'Los Angeles', 'USA', 3979000),
(3, 'Toronto', 'CAN', 2731000), (4, 'Mumbai', 'IND', 12442000),
(5, 'Delhi', 'IND', 11034000), (6, 'Paris', 'FRA', 2140000),
(7, 'Berlin', 'DEU', 3645000), (8, 'Beijing', 'CHN', 21540000);

INSERT INTO countrylanguage VALUES
('USA', 'English', 'T'), ('CAN', 'English', 'T'), ('CAN', 'French', 'T'),
('IND', 'Hindi', 'T'), ('FRA', 'French', 'T'), ('DEU', 'German', 'T');
""")

def run_query(q_no, description, query):
    print(f"Question {q_no}: {description}")
    df = pd.read_sql_query(query, conn)
    print(df.to_string(index=False))
    print("-" * 60 + "\n")

# --- ANSWERS ---

# Q1: Count cities in each country [cite: 40]
run_query(1, "Count how many cities are there in each country?",
          "SELECT CountryCode, COUNT(*) AS CityCount FROM city GROUP BY CountryCode")

# Q2: Continents with > 30 countries [cite: 41]
# Note: Sample data is small, so we use > 1 for demonstration
run_query(2, "Display all continents having more than 30 countries (using > 1 for sample)",
          "SELECT Continent FROM country GROUP BY Continent HAVING COUNT(*) > 1")

# Q3: Regions with population > 200 million [cite: 42]
run_query(3, "List regions whose total population exceeds 200 million",
          "SELECT Region FROM country GROUP BY Region HAVING SUM(Population) > 200000000")

# Q4: Top 5 continents by average GNP [cite: 43]
run_query(4, "Find the top 5 continents by average GNP per country",
          "SELECT Continent, AVG(GNP) as AvgGNP FROM country GROUP BY Continent ORDER BY AvgGNP DESC LIMIT 5")

# Q5: Total official languages per continent [cite: 44]
run_query(5, "Find total number of official languages spoken in each continent",
          """SELECT c.Continent, COUNT(cl.Language) as OfficialLangs
             FROM country c JOIN countrylanguage cl ON c.Code = cl.CountryCode
             WHERE cl.IsOfficial = 'T' GROUP BY c.Continent""")

# Q6: Max and Min GNP for each continent [cite: 45]
run_query(6, "Find the maximum and minimum GNP for each continent",
          "SELECT Continent, MAX(GNP) as MaxGNP, MIN(GNP) as MinGNP FROM country GROUP BY Continent")

# Q7: Country with highest average city population [cite: 46]
run_query(7, "Find the country with the highest average city population",
          """SELECT c.Name, AVG(ci.Population) as AvgCityPop
             FROM country c JOIN city ci ON c.Code = ci.CountryCode
             GROUP BY c.Name ORDER BY AvgCityPop DESC LIMIT 1""")

# Q8: Continents where average city population > 200,000 [cite: 47]
run_query(8, "List continents where average city population is greater than 200,000",
          """SELECT c.Continent FROM country c JOIN city ci ON c.Code = ci.CountryCode
             GROUP BY c.Continent HAVING AVG(ci.Population) > 200000""")

# Q9: Total population and average life expectancy per continent [cite: 48]
run_query(9, "Total population and average life expectancy for each continent",
          """SELECT Continent, SUM(Population) as TotalPop, AVG(LifeExpectancy) as AvgLife
             FROM country GROUP BY Continent ORDER BY AvgLife DESC""")

# Q10: Top 3 continents with high life expectancy and pop > 200M [cite: 49]
run_query(10, "Top 3 continents with highest life expectancy (where Population > 200M)",
          """SELECT Continent, AVG(LifeExpectancy) as AvgLife
             FROM country GROUP BY Continent HAVING SUM(Population) > 200000000
             ORDER BY AvgLife DESC LIMIT 3""")

Question 1: Count how many cities are there in each country?
CountryCode  CityCount
        CAN          1
        CHN          1
        DEU          1
        FRA          1
        IND          2
        USA          2
------------------------------------------------------------

Question 2: Display all continents having more than 30 countries (using > 1 for sample)
    Continent
         Asia
       Europe
North America
------------------------------------------------------------

Question 3: List regions whose total population exceeds 200 million
       Region
 Eastern Asia
North America
South America
Southern Asia
------------------------------------------------------------

Question 4: Find the top 5 continents by average GNP per country
    Continent     AvgGNP
North America 11540000.0
         Asia  8608500.0
       Europe  3288000.0
South America  1847000.0
------------------------------------------------------------

Question 5: Find total number of official languages spoken